# Sector Input-Output Sankey

This notebook uses local Eora-style Czech Republic sector data to show how detailed primary sectors flow into downstream sectors as intermediate inputs.

Numeric source: `CZE_Z_intermediate_2022_labeled.csv`. Sector labels: `CZE_sector_codebook.csv`. Eora26 classification context: `Eora26Structure.xlsx`.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

DATA_DIR = Path("/Users/pstaif/Desktop/Econ Ultimate Map/data")
EORA_STRUCTURE_PATH = DATA_DIR / "Eora26Structure.xlsx"
FLOW_MATRIX_PATH = DATA_DIR / "CZE_Z_intermediate_2022_labeled.csv"
CODEBOOK_PATH = DATA_DIR / "CZE_sector_codebook.csv"

COUNTRY = "Czech Republic"
COUNTRY_CODE = "CZE"
YEAR = 2022
TOP_TARGETS = 15

pd.set_option("display.max_colwidth", 120)

In [ ]:
for path in [EORA_STRUCTURE_PATH, FLOW_MATRIX_PATH, CODEBOOK_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

matrix = pd.read_csv(FLOW_MATRIX_PATH)
codebook = pd.read_csv(CODEBOOK_PATH)
codebook["code"] = codebook["code"].astype(str).str.strip()
codebook["description"] = codebook["description"].astype(str).str.strip()
codebook_map = dict(zip(codebook["code"], codebook["description"]))

print(f"Loaded flow matrix: {matrix.shape[0]} source sectors x {matrix.shape[1] - 1} target sectors")
print(f"Loaded codebook labels: {len(codebook_map)}")
print(f"Eora26 structure workbook found: {EORA_STRUCTURE_PATH.name}")
matrix.head()

In [ ]:
def local_sector_code(full_code):
    text = str(full_code).strip()
    return text.split("_", 1)[1] if "_" in text else text


def label_for_code(full_code):
    local_code = local_sector_code(full_code)
    description = codebook_map.get(local_code, local_code)
    return f"{full_code} | {description}"


def parse_target_header(header):
    text = str(header).strip()
    if " | " in text:
        code, description = text.split(" | ", 1)
        return code.strip(), description.strip()
    return text, codebook_map.get(local_sector_code(text), text)


primary_sector_codes = ["A01", "A02", "A03", "B05", "B06", "B07", "B08", "B09"]
primary_sectors = codebook[codebook["code"].isin(primary_sector_codes)].copy()
primary_sectors

In [ ]:
def build_source_flows(source_code):
    source_rows = matrix[matrix.iloc[:, 0].astype(str).str.strip().eq(source_code)]
    if source_rows.empty:
        available = matrix.iloc[:, 0].astype(str).head(20).tolist()
        raise ValueError(f"Source sector {source_code!r} not found. First available sectors: {available}")
    if len(source_rows) > 1:
        raise ValueError(f"Expected one source row for {source_code}, found {len(source_rows)}")

    source_label = label_for_code(source_code)
    source_series = source_rows.iloc[0, 1:].copy()
    flows = pd.DataFrame({"target_header": source_series.index, "value": pd.to_numeric(source_series.values, errors="coerce")})
    targets = flows["target_header"].apply(parse_target_header)
    flows["target_code"] = targets.apply(lambda item: item[0])
    flows["target_label"] = targets.apply(lambda item: item[1])
    flows["target_node"] = flows["target_code"] + " | " + flows["target_label"]
    flows = flows[flows["value"].gt(0)].sort_values("value", ascending=False).reset_index(drop=True)

    if flows.empty:
        raise ValueError(f"No positive target flows found for {source_code}")

    return source_label, flows, flows["value"].sum()


def render_sector_sankey(source_code, output_html, source_display_name=None):
    output_html = Path(output_html)
    source_label, flows, total_positive_flow = build_source_flows(source_code)

    top_flows = flows.head(TOP_TARGETS).copy()
    other_value = flows.iloc[TOP_TARGETS:]["value"].sum()
    if other_value > 0:
        top_flows = pd.concat(
            [
                top_flows,
                pd.DataFrame(
                    [{"target_code": "OTHER", "target_label": "Other sectors", "target_node": "Other sectors", "value": other_value}]
                ),
            ],
            ignore_index=True,
        )

    labels = [source_label] + top_flows["target_node"].tolist()
    sources = [0] * len(top_flows)
    targets = list(range(1, len(labels)))
    values = top_flows["value"].tolist()

    assert len(labels) > 2, "Sankey needs one source and multiple target sectors."
    assert all(value > 0 for value in values), "All Sankey link values must be positive."
    assert round(sum(values), 6) == round(total_positive_flow, 6), "Top flows plus Other sectors must equal the source row total."

    target_colors = ["#2A9D8F"] * (len(labels) - 1)
    if top_flows.iloc[-1]["target_code"] == "OTHER":
        target_colors[-1] = "#8A817C"

    title_name = source_display_name or source_label.split(" | ", 1)[-1]
    fig = go.Figure(
        data=[
            go.Sankey(
                arrangement="snap",
                node=dict(
                    pad=18,
                    thickness=22,
                    line=dict(color="rgba(39, 39, 39, 0.35)", width=0.5),
                    label=labels,
                    color=["#4F772D"] + target_colors,
                ),
                link=dict(
                    source=sources,
                    target=targets,
                    value=values,
                    color="rgba(143, 188, 143, 0.45)",
                    customdata=top_flows["target_code"],
                    hovertemplate="%{source.label} -> %{target.label}<br>Flow: %{value:,.2f}<extra>%{customdata}</extra>",
                ),
            )
        ]
    )

    fig.update_layout(
        title=(
            f"{COUNTRY} Intermediate Sector Flows: {title_name} to Downstream Sectors "
            f"({YEAR}, local Eora-style matrix)"
        ),
        font=dict(size=13, family="Arial", color="#2f2a25"),
        paper_bgcolor="#f7f3ea",
        margin=dict(l=20, r=20, t=80, b=20),
        height=760,
    )

    output_html.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(output_html, include_plotlyjs="cdn")

    print(f"Source sector: {source_label}")
    print(f"Positive downstream targets: {len(flows)}")
    print(f"Total positive intermediate flow: {total_positive_flow:,.2f}")
    print(f"Saved {output_html}")
    display(flows.head(20))
    fig.show()
    return flows

In [ ]:
agriculture_flows = render_sector_sankey(
    "CZE_A01",
    "output/sector_sankey.html",
    "Agriculture, hunting and related service activities",
)

In [ ]:
forestry_flows = render_sector_sankey(
    "CZE_A02",
    "output/sector_sankey_forestry.html",
    "Forestry and logging",
)

In [ ]:
fishing_flows = render_sector_sankey(
    "CZE_A03",
    "output/sector_sankey_fishing.html",
    "Fishing and aquaculture",
)

In [ ]:
coal_lignite_flows = render_sector_sankey(
    "CZE_B05",
    "output/sector_sankey_coal_lignite.html",
    "Mining of coal and lignite",
)

In [ ]:
oil_gas_flows = render_sector_sankey(
    "CZE_B06",
    "output/sector_sankey_oil_gas.html",
    "Extraction of crude petroleum and natural gas",
)

In [ ]:
metal_ores_flows = render_sector_sankey(
    "CZE_B07",
    "output/sector_sankey_metal_ores.html",
    "Mining of metal ores",
)

In [ ]:
other_mining_flows = render_sector_sankey(
    "CZE_B08",
    "output/sector_sankey_other_mining.html",
    "Other mining and quarrying",
)

In [ ]:
mining_support_flows = render_sector_sankey(
    "CZE_B09",
    "output/sector_sankey_mining_support.html",
    "Mining support service activities",
)